# Week 8: ES Mechanism Ablation for LLM Fine-Tuning

This notebook walks through our ES fine-tuning pipeline in three stages:

1. **CPU demo** — `sanity_es_loop.py`, runs on any machine, shows the core loop
2. **GPU training** — `train_es.py`, all flags, forward-pass budget tracking
3. **Structured experiments** — `run_experiment.py` + `plot_results.py`, all comparison blocks

**Requirements:** `uv sync` from the repo root. No GPU needed for §1.

---
## §1 — CPU Demo: `sanity_es_loop.py`

This is the reference implementation: readable, dependency-minimal, and runnable on a laptop CPU. It is intentionally simple — no GPU flags, no experiment orchestration — to make the algorithm transparent.

### What it does

For each iteration:
1. Sample a fixed mini-batch from the training set.
2. For each of `population_size` seeds, run the **antithetic 3-pass** perturbation:
   - Perturb weights $\theta \to \theta + \sigma\varepsilon$, evaluate batch → $r^+$
   - Shift directly to $\theta - \sigma\varepsilon$, evaluate batch → $r^-$
   - Restore $\theta$
3. Collect advantages $a_k = r^+_k - r^-_k$.
4. z-score normalize advantages, apply ES gradient update.
5. Evaluate on validation set every `val_every` iterations.

In [ ]:
# Inspect the core loop — no execution needed
import inspect
from src.scripts.sanity_es_loop import run_es_iteration, validate
print(inspect.getsource(run_es_iteration))

In [ ]:
from src.utils.perturb import perturb_inplace, restore_inplace, es_grad_update
print(inspect.getsource(perturb_inplace))

### Run the CPU demo

The command below runs 4 ES iterations on RTE with OPT-350M on CPU. Expected wall-clock: **5–15 minutes** on a modern laptop (each forward pass of OPT-350M on CPU takes ~0.5–2s).

Use `--num-iters 2 --population-size 4 --train-size 16 --val-size 50` for a quick smoke test (~2–5 min).

In [ ]:
# Quick smoke test — 2 iters, small data, greedy decode
!uv run python -m src.scripts.sanity_es_loop \
    --task rte \
    --model facebook/opt-350m \
    --num-iters 2 \
    --population-size 4 \
    --batch-size 8 \
    --train-size 16 \
    --val-size 50 \
    --sigma 3e-4 \
    --lr 3e-3 \
    --val-every 2 \
    --out-dir results/cpu_demo_rte

In [ ]:
# Read the log to inspect what was recorded
import json
from pathlib import Path

log = Path("results/cpu_demo_rte/log.jsonl")
if log.exists():
    for line in log.read_text().splitlines():
        print(json.loads(line))
else:
    print("Log not found — run the cell above first.")

### Inspect the task registry

Each task (SST-2, RTE, BoolQ) follows the same interface: `load_data`, `build_prompt`, `score`.

In [ ]:
from src.tasks import get_task, available_tasks
print("Available tasks:", available_tasks())

task = get_task("rte")
train_data, val_data = task.load_data(train_size=4, val_size=4, seed=42)

ex = train_data[0]
print("\nExample:")
print(ex)
print("\nPrompt:")
print(task.build_prompt(ex))
print("\nScore('yes', ex):", task.score("yes", ex))
print("Score('no', ex): ", task.score("no", ex))

### Verify roundtrip invariance of perturbation

The core correctness guarantee: perturb then restore must return to the original parameters exactly.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
import src.utils.perturb as perturb_module
perturb_module.VERBOSE = False  # suppress prints

# Use a tiny model for this test
model = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", dtype=torch.float32)

# Save original norm
orig = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}

seed, sigma = 1234, 1e-3
perturb_inplace(model, seed, sigma, sign=+1)
perturb_inplace(model, seed, 2 * sigma, sign=-1)  # +eps → -eps
restore_inplace(model, seed, sigma, sign=-1)        # -eps → 0

max_err = max(
    (p.data - orig[n]).abs().max().item()
    for n, p in model.named_parameters() if n in orig
)
print(f"Max roundtrip error: {max_err:.2e}  (should be < 1e-5 for float32)")
assert max_err < 1e-5, "Roundtrip failed!"
print("Roundtrip invariance: PASS")

---
## §2 — GPU Training: `train_es.py`

`train_es.py` is the production training script. It extends `sanity_es_loop.py` with:
- `--device cuda` / `--dtype bfloat16` for GPU execution
- `--noise-type gaussian|rademacher`
- `--one-sided` (single perturbation direction)
- `--no-normalize` (skip z-score of advantages)
- `--top-k N` (ARS-style: keep only top-k seeds by |advantage|)
- Forward-pass budget tracking: `train_fwd` (training only) and `total_fwd` (includes val)
- Checkpoint save/resume with run state

### GPU setup

OPT-350M requires ~1.4 GB VRAM in bfloat16. OPT-1.3B requires ~2.6 GB. Both fit comfortably on a single consumer GPU (RTX 3090, A100, etc.).

```bash
# Check GPU availability
python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"

# Install CUDA-enabled PyTorch if needed (example for CUDA 12.1):
# uv add torch torchvision --index-url https://download.pytorch.org/whl/cu121
```

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory // 1024**3, "GB")
else:
    print("No GPU — §2 commands will use CPU (slow for large models)")

CUDA available: False
No GPU — §2 commands will use CPU (slow for large models)


### Run a single GPU training job

In [ ]:
# GPU run (set --device cpu if no GPU available, but expect 10-30x slowdown)
!uv run python -m src.scripts.train_es \
    --task rte \
    --model facebook/opt-350m \
    --device cuda \
    --dtype bfloat16 \
    --num-iters 10 \
    --population-size 8 \
    --batch-size 16 \
    --train-size 64 \
    --val-size 277 \
    --sigma 3e-4 \
    --lr 3e-3 \
    --val-every 2 \
    --early-stop-delta 0 \
    --out-dir results/train_es_demo

### ES variant flags — quick examples

```bash
# One-sided (no antithetic pair)
uv run python -m src.scripts.train_es --one-sided --num-iters 20 ...

# Rademacher noise
uv run python -m src.scripts.train_es --noise-type rademacher ...

# Unnormalized update
uv run python -m src.scripts.train_es --no-normalize ...

# ARS-style top-4 out of N=16
uv run python -m src.scripts.train_es --population-size 16 --top-k 4 ...
```

In [ ]:
# Read train_fwd curve from the demo log
import json
from pathlib import Path

log = Path("results/train_es_demo/log.jsonl")
if log.exists():
    iters = [json.loads(l) for l in log.read_text().splitlines()
              if json.loads(l).get("event") == "iter"]
    print(f"{'iter':>4}  {'train_fwd':>10}  {'total_fwd':>10}  {'val_acc':>8}")
    for e in iters:
        val = f"{e['val_acc']:.3f}" if 'val_acc' in e else '   ---'
        print(f"{e['iteration']:>4}  {e['train_fwd']:>10}  {e['total_fwd']:>10}  {val:>8}")
else:
    print("Run the training cell above first.")

### Resume from checkpoint

```bash
uv run python -m src.scripts.train_es \
    --resume-from results/train_es_demo/latest.pt \
    --num-iters 20 \
    --out-dir results/train_es_resumed
```

The `.state.json` file alongside each `.pt` checkpoint stores `iteration`, `best_val_acc`, `cumulative_train_fwd`, and `cumulative_total_fwd` so that resumed runs continue the forward-pass counter correctly.

---
## §3 — Structured Experiments: `run_experiment.py`

This script orchestrates all experiment blocks. Each block runs multiple variants, each with multiple seeds, all as subprocesses calling `train_es.py`. Results are collected into `summary.json`.

### Available blocks

| Block | Description |
|---|---|
| `calibration` | 4×4 sigma/lr grid on RTE — **run this first** |
| `one_vs_two` | One-sided vs two-sided ES at equal train_fwd |
| `noise_type` | Gaussian vs Rademacher noise |
| `normalize` | z-score on vs off |
| `top_k` | All seeds vs ARS top-4, top-8 (N=16) |
| `pop_scaling` | N ∈ {4, 8, 16, 32} at fixed training budget |
| `task_confirm` | Best config from calibration on BoolQ |

### Step 1: Calibration

In [ ]:
# Calibration: 4x4 grid, 1 seed each (quick scan)
# Expected time: ~16 runs × ~15 min/run on GPU ≈ 4 GPU-hours
# Add --n-seeds 3 for full 3-seed calibration
!uv run python -m src.scripts.run_experiment \
    --block calibration \
    --model facebook/opt-350m \
    --device cuda \
    --n-seeds 1 \
    --out-dir results/exp_week8

In [ ]:
# After calibration, identify best sigma/lr from summary.json
import json
from pathlib import Path

summary_path = Path("results/exp_week8/calibration/summary.json")
if summary_path.exists():
    results = json.loads(summary_path.read_text())
    best = max(results, key=lambda r: r["mean_best_val"] or -1)
    print("Best calibration config:")
    print(f"  sigma={best['config']['sigma']}, lr={best['config']['lr']}")
    print(f"  mean_best_val={best['mean_best_val']:.4f}")
    print()
    print("All results (sorted by val_acc):")
    for r in sorted(results, key=lambda x: x["mean_best_val"] or -1, reverse=True):
        print(f"  sigma={r['config']['sigma']:.0e} lr={r['config']['lr']:.0e}  val={r['mean_best_val']}")
else:
    print("Run calibration first.")

In [ ]:
# Plot calibration heatmap
!uv run python -m src.scripts.plot_results \
    --exp-dir results/exp_week8/calibration \
    --heatmap

from IPython.display import Image
heatmap = Path("results/exp_week8/calibration/calibration_heatmap.png")
if heatmap.exists():
    display(Image(str(heatmap)))

### Step 2: Comparison blocks

Update `BEST_SIGMA` and `BEST_LR` from the calibration result above before running.

In [ ]:
# Set best HP from calibration — update these values!
BEST_SIGMA = 3e-4  # <-- update from calibration
BEST_LR    = 3e-3  # <-- update from calibration
MODEL      = "facebook/opt-350m"
OUT_DIR    = "results/exp_week8"

In [ ]:
# Run all comparison blocks (one at a time, or use --block all)
import subprocess, sys

blocks = ["one_vs_two", "noise_type", "normalize", "top_k", "pop_scaling"]

for block in blocks:
    print(f"\n{'='*50}")
    print(f"Running block: {block}")
    print(f"{'='*50}")
    result = subprocess.run(
        [
            sys.executable, "-m", "src.scripts.run_experiment",
            "--block", block,
            "--model", MODEL,
            "--device", "cuda",
            "--n-seeds", "3",
            "--out-dir", OUT_DIR,
        ],
        check=False,
    )
    if result.returncode != 0:
        print(f"[!] Block {block} failed with rc={result.returncode}")

In [ ]:
# Confirm on BoolQ with best calibrated HPs
!uv run python -m src.scripts.run_experiment \
    --block task_confirm \
    --model {MODEL} \
    --device cuda \
    --n-seeds 3 \
    --best-sigma {BEST_SIGMA} \
    --best-lr {BEST_LR} \
    --out-dir {OUT_DIR}

### Step 3: Plot results

In [ ]:
# Generate one plot per block (val_acc vs training forward passes)
!uv run python -m src.scripts.plot_results \
    --exp-dir results/exp_week8 \
    --recursive

# Display generated figures
from IPython.display import Image, display
from pathlib import Path

for fig in sorted(Path("results/exp_week8").rglob("fig.png")):
    print(fig)
    display(Image(str(fig)))

In [ ]:
# Load and print a summary table across all blocks
import json
from pathlib import Path

all_results = []
for summary_file in sorted(Path("results/exp_week8").glob("*/summary.json")):
    results = json.loads(summary_file.read_text())
    all_results.extend(results)

print(f"{'Block':<15} {'Variant':<20} {'Mean val_acc':>12} {'Std':>8} {'Seeds'}")
print("-" * 70)
for r in all_results:
    mv = f"{r['mean_best_val']:.4f}" if r.get('mean_best_val') is not None else "N/A"
    sv = f"{r.get('std_best_val', None):.4f}" if r.get('std_best_val') is not None else "N/A"
    seeds = [f"{v:.3f}" if v is not None else "N/A" for v in r.get("best_per_seed", [])]
    print(f"{r['block']:<15} {r['variant']:<20} {mv:>12} {sv:>8} {seeds}")

---
## Appendix: GPU setup reference

### Environment

```bash
# Install
uv sync
source .venv/bin/activate

# Verify GPU
python -c "import torch; print(torch.cuda.is_available())"
```

### CUDA-enabled PyTorch (if not already installed)

```bash
# CUDA 12.1
uv add torch --index-url https://download.pytorch.org/whl/cu121

# CUDA 11.8
uv add torch --index-url https://download.pytorch.org/whl/cu118
```

### Memory estimates (bfloat16)

| Model | Params | VRAM (weights) | Peak VRAM (ES, batch=16) |
|---|---|---|---|
| OPT-125M | 125M | ~0.25 GB | ~1 GB |
| OPT-350M | 350M | ~0.70 GB | ~2 GB |
| OPT-1.3B | 1.3B | ~2.60 GB | ~5 GB |

Peak VRAM includes model weights + KV cache for `max_new_tokens=4` at `batch_size=16`. No gradient storage (ES is gradient-free). Peak extra memory from perturbation = size of largest single layer (OPT-350M: the embedding matrix, ~50 MB).

### Multi-GPU (not yet implemented)

The current architecture is single-process single-GPU. To scale:
- Spawn N worker processes, each holding a model replica on its own GPU.
- Coordinator samples seeds, dispatches to workers, collects advantages.
- Apply `es_grad_update` on master weights; broadcast fresh `state_dict` to workers.
- See `torch.distributed` or `ray.util.collective` for weight broadcast primitives.